# 06 Final Summary

## 1. Project Overview

This notebook summarises the complete TruckScenes vehicle-motion analysis developed across the previous notebooks.

The project investigates RADAR and LiDAR vehicle-motion information under a common annotation-derived reference-velocity framework. Rather than treating the two sensing modalities as directly equivalent, the analysis separates several aspects of performance, including velocity accuracy, velocity-estimation availability, target-information availability, information amount, information type, and native observation frequency.

The analysis was developed through the following notebooks:

- `00_issue_overview.ipynb` defines the project problem and analysis direction.
- `01_dataset_inspection.ipynb` examines the TruckScenes dataset structure, sensor observations, metadata, and sampling characteristics.
- `02_annotation_reference_velocity.ipynb` establishes the common annotation-derived vehicle reference velocity.
- `03_radar.ipynb` develops and evaluates the RADAR-derived target-velocity baseline.
- `04_lidar.ipynb` develops and evaluates the LiDAR-derived target-velocity baseline.
- `05_comparison.ipynb` compares RADAR and LiDAR under the same reference-interval and reference-speed conditions.
- `06_final_summary.ipynb` summarises the complete workflow, findings, limitations, and full-dataset extension procedure.

Reusable processing logic validated in these notebooks was progressively extracted into the Python modules in `src/`, including `annotation_velocity.py`, `radar_velocity.py`, `lidar_velocity.py`, `sensor_observation.py`, and `comparison.py`.

Validated intermediate and comparison results are stored in `results/validated/`, allowing later notebooks to reuse completed processing results without repeating the earlier sensor-processing stages.

This final notebook summarises the overall workflow, key findings, methodological limitations, and the procedure for extending the validated analysis from the current TruckScenes mini dataset to the full dataset.

## 2. Analysis Workflow

The project follows a staged analysis workflow so that dataset inspection, reference construction, sensor-specific processing, and final comparison remain separate.

### Stage 1: Problem Definition and Dataset Inspection

`00_issue_overview.ipynb` defines the project problem and the main RADAR–LiDAR comparison direction.

`01_dataset_inspection.ipynb` examines the TruckScenes dataset structure and the RADAR/LiDAR observation characteristics required by the later analysis. Reusable inspection and observation logic is stored in `dataset_inspection.py`, `sensor_observation.py`, and `data_io.py`.

### Stage 2: Common Reference Velocity

`02_annotation_reference_velocity.ipynb` constructs the common vehicle-motion reference from consecutive TruckScenes annotations.

The workflow links annotations to sample timestamps, builds consecutive annotation intervals, calculates position changes and time differences, and derives 2D and 3D reference velocity. The 2D ground-plane speed is used as the primary vehicle reference speed.

The validated processing functions are stored in `annotation_velocity.py`.

### Stage 3: Sensor-Specific Velocity Analysis

`03_radar.ipynb` associates RADAR returns with target vehicles and reconstructs global ground-plane 2D target velocity from ego-compensated RADAR radial-velocity constraints collected at both endpoints of each reference interval. The final comparison uses the validated stable RADAR velocity estimates.

The reusable RADAR processing logic is stored in `radar_velocity.py`.

`04_lidar.ipynb` estimates target positions from target-associated LiDAR points in the global coordinate frame and derives target velocity from consecutive same-sensor observations that satisfy the validated point-support requirements.

The reusable LiDAR processing logic is stored in `lidar_velocity.py`.

### Stage 4: Common RADAR–LiDAR Comparison

`05_comparison.ipynb` combines the validated outputs from the reference, RADAR, and LiDAR analyses into one common interval-level comparison dataset.

Each row represents one valid annotation-derived vehicle-motion interval. RADAR and LiDAR results are left-joined to the reference intervals so that unavailable sensor estimates are retained as part of the analysis.

The comparison examines:

- velocity-estimation availability;
- velocity accuracy;
- matched RADAR–LiDAR accuracy;
- performance across reference-speed ranges;
- target-information availability;
- information amount;
- native observation frequency.

The validated comparison logic is stored in `comparison.py`, while the final interval-level comparison table is stored as `results/validated/comparison_df.pkl`.

This staged structure keeps dataset inspection, sensor-specific velocity estimation, and final comparison logic separate and reusable.

## 3. Key Findings

### 3.1 Reference Vehicle Motion

The annotation-based workflow produced 15,743 valid vehicle-motion reference intervals covering 750 unique vehicles.

The annotation-derived 2D ground-plane speed was used as the primary reference quantity throughout the sensor-specific and cross-sensor analyses. The corresponding 3D speed was retained mainly as a diagnostic metric.

### 3.2 RADAR Velocity Performance

The validated RADAR baseline reconstructed global ground-plane 2D target velocity from ego-compensated RADAR radial-velocity constraints.

Stable RADAR velocity estimates were available for 3,016 of the 15,743 reference intervals, corresponding to a velocity-estimation availability of 19.16%.

Across all available RADAR estimates, the median 2D absolute speed error was 0.277 m/s. However, the mean error was substantially larger at 7.429 m/s, indicating a strongly right-skewed error distribution with a relatively small number of large-error estimates.

### 3.3 LiDAR Velocity Performance

The validated LiDAR baseline derived target velocity from changes in LiDAR-estimated global target position between consecutive same-sensor observations.

LiDAR-derived velocity was available for 8,007 reference intervals, corresponding to a velocity-estimation availability of 50.86%.

Across these available estimates, the median 2D absolute speed error was 0.224 m/s and the mean absolute error was 0.510 m/s. The current LiDAR baseline therefore provided broader velocity-estimation coverage and a less extreme upper error tail than the current RADAR baseline.

### 3.4 Direct RADAR–LiDAR Comparison

RADAR and LiDAR velocity estimates were simultaneously available for 2,899 reference intervals. This matched subset provides the most direct accuracy comparison because both methods are evaluated under the same reference intervals.

On the matched subset, the median 2D absolute speed errors were very similar:

- RADAR: 0.266 m/s
- LiDAR: 0.263 m/s

The difference became much larger in the upper error distribution:

- RADAR mean absolute error: 6.455 m/s
- LiDAR mean absolute error: 0.613 m/s
- RADAR 90th-percentile error: 9.558 m/s
- LiDAR 90th-percentile error: 1.437 m/s

These results show that the two methods have similar typical error on the matched intervals, but the current RADAR baseline has a substantially heavier error tail.

Velocity performance also varies across reference-speed ranges. The two methods show similar median errors in the lowest speed ranges, while RADAR exhibits larger and more variable median errors in several medium- and high-speed ranges. These patterns should be interpreted cautiously because the number of matched observations differs substantially between speed ranges.

### 3.5 Sensor Information and Efficiency

Target-associated RADAR information was available at both endpoints for 7,706 reference intervals, corresponding to 48.95% of all valid reference intervals. Dataset-provided LiDAR target information was available at both endpoints for all 15,743 intervals.

The corresponding velocity-estimation availability was lower for both methods:

- RADAR: 19.16%
- LiDAR: 50.86%

Among intervals where target information was available, 39.14% produced a stable RADAR velocity estimate and 50.86% produced a valid LiDAR-derived velocity estimate under the current methods.

The median interval-level information amount was 7 RADAR returns and 90 LiDAR points. These values are not directly comparable because RADAR returns and LiDAR points are different information units.

Native observation frequency provides a different efficiency characteristic. RADAR operates at approximately 19.5 Hz in the current dataset, compared with approximately 10 Hz for LiDAR. RADAR therefore provides more frequent native observations, while the current LiDAR analysis provides broader target-information and velocity-estimation coverage at the common reference intervals.

Overall, the results indicate different sensing trade-offs rather than a single universal performance ranking between RADAR and LiDAR.

## 4. Reusability and Full-Dataset Workflow

The current analysis was developed and validated using the TruckScenes mini dataset, but the processing workflow was structured so that the same analysis logic can be reused with the complete dataset.

### 4.1 Reusable Project Structure

Dataset-specific file locations are separated from the processing logic through `config.py`. The current configuration defines the dataset root and metadata directory used by the analysis.

The main reusable modules are:

- `data_io.py` for basic JSON and PCD loading;
- `dataset_inspection.py` for dataset and sensor-structure inspection;
- `sensor_observation.py` for RADAR/LiDAR observation metadata, sampling intervals, and frequency analysis;
- `annotation_velocity.py` for annotation-derived reference velocity;
- `radar_velocity.py` for target-associated RADAR processing and RADAR-derived 2D target velocity;
- `lidar_velocity.py` for target-associated LiDAR processing and LiDAR-derived target velocity;
- `comparison.py` for building the common interval-level RADAR–LiDAR comparison dataset.

These modules contain the processing methods validated in the previous notebooks and do not depend on the number of scenes, vehicles, or reference intervals in the mini dataset.

### 4.2 Switching to the Full TruckScenes Dataset

To use the complete TruckScenes dataset, the dataset location and metadata version should first be updated in `config.py`.

The current configuration uses:

```python
DATA_ROOT = PROJECT_ROOT / "data" / "man-truckscenes"
METADATA_DIR_NAME = "v1.2-mini"
```

For the full dataset, DATA_ROOT should point to the full TruckScenes installation and METADATA_DIR_NAME should be changed to the metadata directory corresponding to the complete dataset.

The processing functions themselves should not require changes as long as the full dataset follows the same TruckScenes metadata and sensor-file structure.

### 4.3 Full-Dataset Processing Sequence

The full-dataset analysis should follow the same validated sequence:

1. Run 01_dataset_inspection.ipynb to confirm the full dataset structure, sensor channels, metadata, and observation characteristics.
2. Run 02_annotation_reference_velocity.ipynb to regenerate the full annotation-derived reference-velocity dataset.
3. Run 03_radar.ipynb to regenerate target-associated RADAR information and stable RADAR-derived velocity estimates.
4. Run 04_lidar.ipynb to regenerate target-associated LiDAR information and LiDAR-derived velocity estimates.
5. Save the validated full-dataset intermediate results in results/validated/.
6. Run 05_comparison.ipynb using the regenerated full-dataset results to construct the final common comparison dataset.
7. Recalculate all availability, accuracy, speed-range, information, and efficiency statistics using the full dataset.

The previous mini-dataset numerical results should not be reused as full-dataset results. Only the validated processing logic, comparison definitions, and analysis structure are intended to be reused.

### 4.4 Expected Reusability

The transition from the mini dataset to the full dataset therefore requires the sensor and reference data to be processed again, but it should not require the analysis methods to be redesigned.

The main expected workflow is:

full TruckScenes data
→ reference-velocity processing
→ RADAR processing
→ LiDAR processing
→ validated intermediate results
→ common comparison
→ full-dataset findings

The larger dataset may increase processing time and memory requirements, particularly during point-level RADAR and LiDAR processing. However, the comparison stage operates primarily on target-level and interval-level processed tables rather than directly on the complete raw point-cloud data.

The full-dataset analysis should therefore be treated as a rerun of the validated pipeline on a larger dataset rather than a new analysis methodology.

## 5. Final Conclusions and Limitations

### 5.1 Final Conclusions

This project established a complete TruckScenes workflow for comparing RADAR and LiDAR vehicle-motion information under a common annotation-derived reference-velocity framework.

The final comparison shows that the two sensing modalities have different strengths and limitations rather than one sensor being universally better.

Under the current validated methods:

- LiDAR provides broader target-information coverage at the common reference intervals.
- LiDAR also provides higher velocity-estimation availability than the stable RADAR baseline.
- On the matched subset where both velocity estimates are available, the two methods have very similar median 2D speed error.
- The current RADAR baseline has a substantially heavier upper error tail, while the LiDAR baseline shows more consistent matched-error behaviour.
- RADAR provides a higher native observation frequency and direct velocity-related measurements.
- LiDAR provides dense spatial target information, but its vehicle velocity must be derived from consecutive target-position estimates.

These results show that velocity accuracy, velocity availability, information availability, information amount, information type, and observation frequency should be considered separately when comparing RADAR and LiDAR.

A single efficiency score would hide important differences between the sensing modalities. The current analysis therefore treats sensor efficiency as a combination of complementary characteristics rather than one numerical ranking.

The project also produced a reusable processing structure. The validated analysis logic is separated into Python modules, and the same workflow can be rerun on the complete TruckScenes dataset by updating the dataset configuration and regenerating the intermediate results.

### 5.2 Limitations

Several limitations remain in the current analysis.

- The current results are based on the TruckScenes mini dataset. The numerical findings should therefore be re-evaluated using the complete dataset before drawing broader conclusions.
- The annotation-derived reference velocity is calculated from consecutive annotated target positions and is therefore a derived reference rather than an independent direct velocity measurement.
- The main cross-sensor velocity comparison is limited to 2D ground-plane speed because a directly comparable validated RADAR 3D velocity estimate was not established.
- RADAR and LiDAR velocity availability depends on the current estimation methods. The reported availability values should not be interpreted as intrinsic sensor availability.
- The RADAR baseline requires sufficient radial-constraint geometry and numerical stability, while the LiDAR baseline requires consecutive same-sensor observations with sufficient target-associated point support.
- RADAR and LiDAR information amounts are based on different measurement units. A RADAR return and a LiDAR point cannot be treated as equivalent information units.
- The LiDAR information amount uses the dataset-provided target-associated LiDAR point count, while RADAR information amount is based on target-associated RADAR returns obtained by the validated association workflow.
- The main comparison is performed at annotation/keyframe intervals and therefore does not fully exploit the higher native RADAR observation frequency.
- The number of matched intervals differs across reference-speed ranges, particularly in several higher-speed ranges. Observed speed-dependent patterns should therefore not be interpreted as causal effects.
- Large RADAR errors were retained in the analysis rather than removed as outliers. This preserves the behaviour of the validated baseline but produces a large difference between median and mean RADAR error.

_note: During development, additional validation, diagnostic, and method-comparison code was used to verify intermediate assumptions and select the final processing choices. To keep the final notebooks concise and focused on the validated workflow, not all exploratory and intermediate validation code has been retained in the submitted notebook sequence._

Overall, the current TruckScenes analysis suggests that LiDAR provides broader and more consistent vehicle-motion estimation under the validated keyframe-level workflow, while RADAR provides more frequent native observations and direct velocity-related sensing information.

The final interpretation is therefore based on sensing trade-offs rather than a universal ranking between RADAR and LiDAR.